# LDA & SVM Odor Classification Analysis

**Purpose:** Compare classifiers for odor identity discrimination

**Methods Tested:**
1. **LDA (Linear Discriminant Analysis)** - Linear boundary, dimensionality reduction
2. **SVM (Support Vector Machine)** - Maximum margin classifier


**Analysis:**
- 7 odor stimuli classification
- Cross-validation + permutation testing
- Fisher's method + FDR correction
- **Confusion matrix analysis** 

---

## Notebook Structure:
1. Setup & Configuration
2. Helper Functions (including confusion matrix utilities)
3. LDA Classification Analysis
4. LDA Confusion Matrix Analysis
5. SVM Classification Analysis  
6. SVM Confusion Matrix Analysis
7. Method Comparison 
8. Summary


## 1. Setup & Configuration


In [ ]:
# Imports
import numpy as np
import pandas as pd
import json
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.utils import shuffle
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful")


### Configuration


In [ ]:
# ============================================================================
# FILE PATHS
# ============================================================================
NEURAL_DATA_PATH = '/Volumes/LaCie/larval_HuC/imaging/7dpf_wb_population_general_response_vector.h5'
OUTPUT_DIR = '/Volumes/LaCie/larval_HuC/imaging'

# ============================================================================
# FISH IDS
# ============================================================================
FISH_IDS = [
    ['230713_fb', '230713_hb'], ['230714_fb', '230714_hb'],
    ['230720_fb', '230720_hb'], ['230727_fb', '230727_hb'],
    ['230728_fb', '230728_hb'], ['230810_fb', '230810_hb'],
    ['230811_fb', '230811_hb'], ['230817_fb', '230817_hb'],
    ['230818_fb', '230818_hb'], ['230919_fb', '230919_hb']
]

# ============================================================================
# STIMULUS PARAMETERS
# ============================================================================
KEY_STIMULI = ('ade', 'pro_25um', 'qui_25um', 'cad_25um', 'fex_3', 'kw', 'ph4.5')

# Valence categories (for confusion analysis)
ATTRACTIVE = ['fex_3', 'kw', 'pro_25um']
AVERSIVE = ['ade', 'cad_25um', 'qui_25um', 'ph4.5']

# ============================================================================
# CLASSIFIER PARAMETERS
# ============================================================================
# LDA parameters
N_PCA_COMPONENTS = 5       # PCA dimensions before LDA
N_FOLDS_LDA = 3            # Cross-validation folds

# SVM parameters  
N_FOLDS_SVM = 3            # Cross-validation folds

# Shared parameters
MIN_NEURONS = 5            # Minimum neurons per area
N_PERMUTATIONS = 1000      # Permutation test iterations
FDR_ALPHA = 0.05           # FDR significance threshold

print(f"✓ Configuration loaded")
print(f"  Fish: {len(FISH_IDS)}")
print(f"  Stimuli: {len(KEY_STIMULI)}")
print(f"  PCA components: {N_PCA_COMPONENTS}")
print(f"  Permutations: {N_PERMUTATIONS}")


## 2. Helper Functions

Modular functions for:
- Classification (LDA/SVM)
- Permutation testing
- Statistical combination
- **Confusion matrix visualization (template matching style)**


### Classification Functions


In [ ]:
def run_lda_classification(X, y, n_pca=5, n_folds=3):
    """Run LDA classification with PCA preprocessing.
    
    Pipeline: PCA → LDA → Cross-validation
    
    Args:
        X: Feature matrix [samples × features]
        y: Labels
        n_pca: Number of PCA components
        n_folds: CV folds
        
    Returns:
        dict with 'mean_score', 'cv_scores', 'confusion_matrix'
    """
    # PCA dimensionality reduction
    pca = PCA(n_components=n_pca, random_state=42)
    X_pca = pca.fit_transform(X)
    
    # LDA classifier
    lda = LinearDiscriminantAnalysis()
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    cv_scores = cross_val_score(lda, X_pca, y, cv=cv)
    
    # Fit on full data for confusion matrix
    lda.fit(X_pca, y)
    y_pred = lda.predict(X_pca)
    
    # Calculate confusion matrix
    unique_labels = sorted(set(y))
    cm = confusion_matrix(y, y_pred, labels=unique_labels)
    
    return {
        'mean_score': np.mean(cv_scores),
        'cv_scores': cv_scores,
        'confusion_matrix': cm,
        'labels': unique_labels
    }

print("✓ run_lda_classification() defined")


In [ ]:
def run_svm_classification(X, y, n_folds=3):
    """Run SVM classification with cross-validation.
    
    Args:
        X: Feature matrix
        y: Labels  
        n_folds: CV folds
        
    Returns:
        dict with 'mean_score', 'cv_scores', 'confusion_matrix'
    """
    # LinearSVC classifier
    svm = LinearSVC(random_state=42, max_iter=10000)
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    cv_scores = cross_val_score(svm, X, y, cv=cv)
    
    # Fit on full data for confusion matrix
    svm.fit(X, y)
    y_pred = svm.predict(X)
    
    # Calculate confusion matrix
    unique_labels = sorted(set(y))
    cm = confusion_matrix(y, y_pred, labels=unique_labels)
    
    return {
        'mean_score': np.mean(cv_scores),
        'cv_scores': cv_scores,
        'confusion_matrix': cm,
        'labels': unique_labels
    }

print("✓ run_svm_classification() defined")


### Permutation Testing


In [ ]:
def permutation_test_classifier(X, y, classifier_func, n_permutations=1000, **kwargs):
    """Permutation test for classifier significance.
    
    Args:
        X: Feature matrix
        y: True labels
        classifier_func: Function that returns {'mean_score': ...}
        n_permutations: Number of shuffles
        **kwargs: Additional arguments for classifier_func
        
    Returns:
        p_value
    """
    # Real accuracy
    real_result = classifier_func(X, y, **kwargs)
    real_accuracy = real_result['mean_score']
    
    # Null distribution
    null_accuracies = []
    for _ in range(n_permutations):
        shuffled_y = shuffle(y, random_state=None)
        perm_result = classifier_func(X, shuffled_y, **kwargs)
        null_accuracies.append(perm_result['mean_score'])
    
    # P-value
    p_value = np.sum(np.array(null_accuracies) >= real_accuracy) / n_permutations
    
    return p_value

print("✓ permutation_test_classifier() defined")


### Statistical Combination


In [ ]:
def combine_fish_statistics(all_areas_dict, fdr_alpha=0.05):
    """Combine p-values using Fisher's method + FDR correction.
    
    Args:
        all_areas_dict: Dict[area] = [(fish_id, score, p_value), ...]
        fdr_alpha: FDR threshold
        
    Returns:
        DataFrame with combined statistics
    """
    combined_stats = {}
    
    for area, fish_results in all_areas_dict.items():
        p_values = [p for _, _, p in fish_results]
        scores = [s for _, s, _ in fish_results]
        
        try:
            _, combined_p = combine_pvalues(p_values, method='fisher')
        except:
            combined_p = 1.0
        
        combined_stats[area] = {
            'combined_p': combined_p,
            'mean_accuracy': np.mean(scores),
            'std_accuracy': np.std(scores),
            'n_fish': len(fish_results)
        }
    
    # FDR correction
    areas = list(combined_stats.keys())
    p_array = [combined_stats[a]['combined_p'] for a in areas]
    
    _, p_corrected, _, _ = multipletests(p_array, alpha=fdr_alpha, method='fdr_bh')
    
    for area, p_fdr in zip(areas, p_corrected):
        combined_stats[area]['fdr_corrected'] = p_fdr
        combined_stats[area]['is_significant'] = p_fdr < fdr_alpha
    
    # Convert to DataFrame
    records = []
    for area, stats in combined_stats.items():
        records.append({
            'area': area,
            'mean_accuracy': stats['mean_accuracy'],
            'std_accuracy': stats['std_accuracy'],
            'combined_p': stats['combined_p'],
            'fdr_corrected': stats['fdr_corrected'],
            'is_significant': stats['is_significant'],
            'n_fish': stats['n_fish']
        })
    
    return pd.DataFrame(records).sort_values('fdr_corrected')

print("✓ combine_fish_statistics() defined")


### Confusion Matrix Utilities (Template Matching Style)


In [ ]:
def plot_confusion_matrix_styled(cm, labels, title="Confusion Matrix", ax=None):
    """Plot confusion matrix in template matching style.
    
    Features:
    - Normalized heatmap
    - Text annotations
    - Gridlines
    - Blue colormap
    
    Args:
        cm: Confusion matrix (raw counts)
        labels: Label names
        title: Plot title
        ax: Matplotlib axis (optional)
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    # Normalize by row
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    
    # Plot
    im = ax.imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    
    # Labels
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=10)
    ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel('Predicted Stimulus', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Stimulus', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Text annotations
    for i in range(len(labels)):
        for j in range(len(labels)):
            color = "white" if cm_norm[i, j] > 0.5 else "black"
            ax.text(j, i, f'{cm_norm[i, j]:.2f}',
                   ha="center", va="center", color=color,
                   fontsize=9, fontweight='bold')
    
    # Gridlines
    ax.set_xticks(np.arange(len(labels)) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(labels)) - 0.5, minor=True)
    ax.grid(which="minor", color="gray", linestyle='-', linewidth=0.5)
    
    return im

print("✓ plot_confusion_matrix_styled() defined")


In [ ]:
def analyze_confusion_pairs(cm, labels, attractive=ATTRACTIVE, aversive=AVERSIVE):
    """Analyze confusion patterns (template matching style).
    
    Returns:
        dict with top_confusions, within_valence_avg, cross_valence_avg
    """
    # Normalize
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    
    # Find confusion pairs (off-diagonal)
    confusion_pairs = []
    for i in range(len(labels)):
        for j in range(len(labels)):
            if i != j:
                confusion_pairs.append((
                    labels[i], labels[j], cm_norm[i, j]
                ))
    
    # Sort by probability
    confusion_pairs.sort(key=lambda x: x[2], reverse=True)
    
    # Categorize by valence
    within_attr = [p for p in confusion_pairs 
                   if p[0] in attractive and p[1] in attractive]
    within_aver = [p for p in confusion_pairs 
                   if p[0] in aversive and p[1] in aversive]
    cross = [p for p in confusion_pairs 
             if (p[0] in attractive and p[1] in aversive) or
                (p[0] in aversive and p[1] in attractive)]
    
    return {
        'top_confusions': confusion_pairs[:10],
        'within_attractive': within_attr,
        'within_aversive': within_aver,
        'cross_valence': cross,
        'within_attr_avg': np.mean([p[2] for p in within_attr]) if within_attr else 0,
        'within_aver_avg': np.mean([p[2] for p in within_aver]) if within_aver else 0,
        'cross_avg': np.mean([p[2] for p in cross]) if cross else 0
    }

print("✓ analyze_confusion_pairs() defined")


### Visualization Utilities


In [ ]:
def jitter(values, amount=0.05):
    """Add random jitter for visualization."""
    return values + np.random.uniform(-amount, amount, len(values))

print("✓ jitter() defined")
print("\n" + "="*80)
print("All helper functions loaded!")
print("="*80)


## 3. LDA Classification Analysis

**Method:** Linear Discriminant Analysis with PCA preprocessing

**Pipeline:**
1. PCA (reduce to 5 dimensions)
2. LDA classifier
3. Cross-validation (3 folds)
4. Permutation testing (1000 shuffles)
5. Statistical combination (Fisher's + FDR)

**For each fish and brain area:**
- Classify 7 odors from neural population patterns
- Test significance
- Collect confusion matrices


### Initialize Storage


In [ ]:
# Storage for LDA results
all_lda_results = {}
all_lda_confusion_matrices = {}

print(f"Starting LDA analysis...")
print(f"  Fish: {len(FISH_IDS)}")
print(f"  Stimuli: {len(KEY_STIMULI)}")
print(f"  PCA components: {N_PCA_COMPONENTS}")
print(f"  CV folds: {N_FOLDS_LDA}")


### Main LDA Processing Loop


In [ ]:
# Process each fish
for fish_num, (fb_id, hb_id) in enumerate(FISH_IDS, 1):
    print(f"\n{'='*80}")
    print(f"Fish {fish_num}/{len(FISH_IDS)}: {fb_id}, {hb_id}")
    print(f"{'='*80}")
    
    # Load data
    data_fb = pd.read_hdf(NEURAL_DATA_PATH, where=f'fish_id == "{fb_id}"')
    data_hb = pd.read_hdf(NEURAL_DATA_PATH, where=f'fish_id == "{hb_id}"')
    data = pd.concat([data_fb, data_hb])
    
    # Filter for key stimuli
    stimulus_df = data[data['stimulus'].isin(KEY_STIMULI)].copy()
    
    # Parse coordinates
    stimulus_df['coords'] = stimulus_df['coords_serialised'].apply(
        lambda x: np.asarray(json.loads(x))
    )
    stimulus_df.drop('coords_serialised', axis=1, inplace=True)
    
    unique_areas = stimulus_df['area'].unique()
    fish_results = {}
    fish_conf_matrices = {}
    
    print(f"  Found {len(unique_areas)} brain areas")
    
    # Process each area
    for area_num, area in enumerate(unique_areas, 1):
        area_df = stimulus_df.query("area == @area")
        
        # Pivot to matrix
        df_resp = area_df.pivot_table(
            index=['fish_id', 'neuron_id'],
            columns=['stimulus', 'trial_number'],
            values='resp'
        )
        
        # Get stimulus labels
        stimulus_names = [col[0] for col in df_resp.columns]
        
        # Remove NaN neurons
        resp_clean = df_resp.dropna()
        
        if len(resp_clean) < MIN_NEURONS:
            continue
        
        print(f"    [{area_num}/{len(unique_areas)}] {area}: {len(resp_clean)} neurons")
        
        # Prepare data
        X = resp_clean.values.T  # Transpose: trials × neurons
        y = np.array(stimulus_names)
        
        # Run LDA classification
        lda_result = run_lda_classification(
            X, y, 
            n_pca=N_PCA_COMPONENTS, 
            n_folds=N_FOLDS_LDA
        )
        mean_accuracy = lda_result['mean_score']
        
        # Permutation test
        p_value = permutation_test_classifier(
            X, y,
            classifier_func=run_lda_classification,
            n_permutations=N_PERMUTATIONS,
            n_pca=N_PCA_COMPONENTS,
            n_folds=N_FOLDS_LDA
        )
        
        # Store results
        fish_results[area] = {
            'mean_score': mean_accuracy,
            'p_value': p_value,
            'n_neurons': len(resp_clean)
        }
        
        fish_conf_matrices[area] = {
            'confusion_matrix': lda_result['confusion_matrix'],
            'labels': lda_result['labels']
        }
        
        print(f"      Accuracy: {mean_accuracy:.3f}, p-value: {p_value:.4f}")
    
    # Store fish results
    fish_key = f"{fb_id}_{hb_id}"
    all_lda_results[fish_key] = fish_results
    all_lda_confusion_matrices[fish_key] = fish_conf_matrices
    
    print(f"  Completed {len(fish_results)} areas")

print(f"\n{'='*80}")
print(f"✓ LDA analysis complete")
print(f"{'='*80}")


### Statistical Combination


In [ ]:
# Reorganize by area
all_areas_lda = {}

for fish_id, fish_results in all_lda_results.items():
    for area_name, area_results in fish_results.items():
        mean_score = area_results['mean_score']
        p_value = area_results['p_value']
        
        if area_name not in all_areas_lda:
            all_areas_lda[area_name] = []
        
        all_areas_lda[area_name].append((fish_id, mean_score, p_value))

print(f"\nCombining LDA statistics...")
print(f"  Total areas: {len(all_areas_lda)}")

# Fisher's + FDR
df_lda_combined = combine_fish_statistics(all_areas_lda, fdr_alpha=FDR_ALPHA)

# Display
print(f"\n{'='*80}")
print("LDA CLASSIFICATION RESULTS")
print(f"{'='*80}")
sig_areas = df_lda_combined[df_lda_combined['is_significant']]
print(f"\nSignificant areas (FDR < {FDR_ALPHA}):")
print(f"  {len(sig_areas)}/{len(df_lda_combined)} areas")

if len(sig_areas) > 0:
    print(f"\nTop 10:")
    print(sig_areas[['area', 'mean_accuracy', 'fdr_corrected']].head(10).to_string(index=False))

print(f"\nAccuracy statistics:")
print(f"  Mean: {df_lda_combined['mean_accuracy'].mean():.3f}")
print(f"  Range: {df_lda_combined['mean_accuracy'].min():.3f} - {df_lda_combined['mean_accuracy'].max():.3f}")
print(f"  Chance: {1/len(KEY_STIMULI):.3f}")


### Save LDA Results


In [ ]:
# Save results
output_file_lda = f"{OUTPUT_DIR}/lda_classification_7odors_refactored.csv"
df_lda_combined.to_csv(output_file_lda, index=False)

print(f"\n✓ LDA results saved to:")
print(f"  {output_file_lda}")
print(f"  Rows: {len(df_lda_combined)}")
print(f"  Significant: {df_lda_combined['is_significant'].sum()}")


## 4. LDA Confusion Matrix Analysis

**Template Matching Style Visualization:**
- Aggregate confusion matrix (all fish & areas)
- Per-fish confusion matrices (2×5 grid)
- Significant areas only
- Confusion pair analysis
- Valence structure analysis


### Aggregate Confusion Matrix


In [ ]:
# Aggregate all LDA confusion matrices
aggregate_lda_cm = np.zeros((len(KEY_STIMULI), len(KEY_STIMULI)))
n_lda_matrices = 0

for fish_id, fish_conf_dict in all_lda_confusion_matrices.items():
    for area, conf_data in fish_conf_dict.items():
        cm = conf_data['confusion_matrix']
        aggregate_lda_cm += cm
        n_lda_matrices += 1

# Average
aggregate_lda_cm_avg = aggregate_lda_cm / n_lda_matrices

print(f"Aggregated {n_lda_matrices} LDA confusion matrices")
print(f"  From {len(all_lda_confusion_matrices)} fish")
print(f"  Across {len(all_areas_lda)} areas")


### Visualize Aggregate LDA Confusion Matrix


In [ ]:
# Plot aggregate
fig, ax = plt.subplots(figsize=(10, 8))

im = plot_confusion_matrix_styled(
    aggregate_lda_cm_avg,
    list(KEY_STIMULI),
    title="LDA: Aggregate Confusion Matrix\n(All Fish & Areas)",
    ax=ax
)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Prediction Probability', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Aggregate LDA confusion matrix plotted")


### Per-Fish LDA Confusion Matrices


In [ ]:
# Per-fish confusion matrices
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

simplified_fish_ids = {f"{fb}_{hb}": f"F{i+1}" 
                      for i, (fb, hb) in enumerate(FISH_IDS)}

for fish_idx, (fish_key, fish_conf_dict) in enumerate(all_lda_confusion_matrices.items()):
    ax = axes[fish_idx]
    
    # Aggregate across areas for this fish
    fish_cm = np.zeros((len(KEY_STIMULI), len(KEY_STIMULI)))
    n_areas = 0
    
    for area, conf_data in fish_conf_dict.items():
        fish_cm += conf_data['confusion_matrix']
        n_areas += 1
    
    # Average
    fish_cm_avg = fish_cm / n_areas
    
    # Plot
    plot_confusion_matrix_styled(
        fish_cm_avg,
        list(KEY_STIMULI),
        title=f"{simplified_fish_ids[fish_key]} (n={n_areas} areas)",
        ax=ax
    )

fig.suptitle('LDA: Per-Fish Confusion Matrices', 
            fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()

print(f"\n✓ Per-fish LDA confusion matrices plotted")


### Significant Areas Only


In [ ]:
# Focus on significant areas
sig_areas_list = df_lda_combined[df_lda_combined['is_significant']]['area'].tolist()

if len(sig_areas_list) > 0:
    sig_lda_cm = np.zeros((len(KEY_STIMULI), len(KEY_STIMULI)))
    n_sig = 0
    
    for fish_id, fish_conf_dict in all_lda_confusion_matrices.items():
        for area, conf_data in fish_conf_dict.items():
            if area in sig_areas_list:
                sig_lda_cm += conf_data['confusion_matrix']
                n_sig += 1
    
    sig_lda_cm_avg = sig_lda_cm / n_sig
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    im = plot_confusion_matrix_styled(
        sig_lda_cm_avg,
        list(KEY_STIMULI),
        title=f"LDA: Significant Areas Only\n({len(sig_areas_list)} areas, FDR < {FDR_ALPHA})",
        ax=ax
    )
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Prediction Probability', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Significant areas LDA confusion matrix")
    print(f"  {n_sig} matrices aggregated")
else:
    print("\nNo significant areas (FDR < 0.05)")


### LDA Confusion Analysis


In [ ]:
# Analyze confusion patterns
print("="*80)
print("LDA CONFUSION ANALYSIS")
print("="*80)

lda_analysis = analyze_confusion_pairs(aggregate_lda_cm_avg, list(KEY_STIMULI))

print("\nTop 10 Confusion Pairs:")
print("-"*80)
for i, (true_stim, pred_stim, prob) in enumerate(lda_analysis['top_confusions'], 1):
    print(f"{i:2d}. {true_stim:12s} → {pred_stim:12s}  (p={prob:.3f})")

print("\n" + "="*80)
print("VALENCE-BASED ANALYSIS")
print("="*80)
print(f"\nWithin-attractive: {lda_analysis['within_attr_avg']:.3f}")
print(f"Within-aversive:   {lda_analysis['within_aver_avg']:.3f}")
print(f"Cross-valence:     {lda_analysis['cross_avg']:.3f}")

if lda_analysis['within_attr_avg'] > lda_analysis['cross_avg'] or \
   lda_analysis['within_aver_avg'] > lda_analysis['cross_avg']:
    print("\n✓ Confusions MORE common within same valence")
    print("  → LDA encodes valence structure")
else:
    print("\n✓ Confusions NOT structured by valence")
    print("  → LDA encodes odor identity independently")


### Save LDA Confusion Matrix


In [ ]:
# Save confusion matrix
cm_lda_file = f"{OUTPUT_DIR}/lda_aggregate_confusion_matrix.csv"
cm_lda_df = pd.DataFrame(
    aggregate_lda_cm_avg,
    index=KEY_STIMULI,
    columns=KEY_STIMULI
)
cm_lda_df.to_csv(cm_lda_file)

print(f"\n✓ LDA confusion matrix saved:")
print(f"  {cm_lda_file}")
print("\n" + "="*80)
print("LDA ANALYSIS COMPLETE")
print("="*80)


## 5. SVM Classification Analysis

**Method:** Support Vector Machine (LinearSVC)

**Pipeline:**
1. LinearSVC classifier (no PCA preprocessing)
2. Cross-validation (3 folds)
3. Permutation testing (1000 shuffles)
4. Statistical combination (Fisher's + FDR)

**Comparison to LDA:**
- SVM: Maximum margin boundary
- No dimensionality reduction
- Different geometric approach


### Initialize Storage


In [ ]:
# Storage for SVM results
all_svm_results = {}
all_svm_confusion_matrices = {}

print(f"Starting SVM analysis...")
print(f"  Fish: {len(FISH_IDS)}")
print(f"  Stimuli: {len(KEY_STIMULI)}")
print(f"  CV folds: {N_FOLDS_SVM}")


### Main SVM Processing Loop


In [ ]:
# Process each fish
for fish_num, (fb_id, hb_id) in enumerate(FISH_IDS, 1):
    print(f"\n{'='*80}")
    print(f"Fish {fish_num}/{len(FISH_IDS)}: {fb_id}, {hb_id}")
    print(f"{'='*80}")
    
    # Load data
    data_fb = pd.read_hdf(NEURAL_DATA_PATH, where=f'fish_id == "{fb_id}"')
    data_hb = pd.read_hdf(NEURAL_DATA_PATH, where=f'fish_id == "{hb_id}"')
    data = pd.concat([data_fb, data_hb])
    
    # Filter for key stimuli
    stimulus_df = data[data['stimulus'].isin(KEY_STIMULI)].copy()
    
    # Parse coordinates
    stimulus_df['coords'] = stimulus_df['coords_serialised'].apply(
        lambda x: np.asarray(json.loads(x))
    )
    stimulus_df.drop('coords_serialised', axis=1, inplace=True)
    
    unique_areas = stimulus_df['area'].unique()
    fish_results = {}
    fish_conf_matrices = {}
    
    print(f"  Found {len(unique_areas)} brain areas")
    
    # Process each area
    for area_num, area in enumerate(unique_areas, 1):
        area_df = stimulus_df.query("area == @area")
        
        # Pivot
        df_resp = area_df.pivot_table(
            index=['fish_id', 'neuron_id'],
            columns=['stimulus', 'trial_number'],
            values='resp'
        )
        
        stimulus_names = [col[0] for col in df_resp.columns]
        resp_clean = df_resp.dropna()
        
        if len(resp_clean) < MIN_NEURONS:
            continue
        
        print(f"    [{area_num}/{len(unique_areas)}] {area}: {len(resp_clean)} neurons")
        
        # Prepare data
        X = resp_clean.values.T
        y = np.array(stimulus_names)
        
        # Run SVM classification
        svm_result = run_svm_classification(X, y, n_folds=N_FOLDS_SVM)
        mean_accuracy = svm_result['mean_score']
        
        # Permutation test
        p_value = permutation_test_classifier(
            X, y,
            classifier_func=run_svm_classification,
            n_permutations=N_PERMUTATIONS,
            n_folds=N_FOLDS_SVM
        )
        
        # Store
        fish_results[area] = {
            'mean_score': mean_accuracy,
            'p_value': p_value,
            'n_neurons': len(resp_clean)
        }
        
        fish_conf_matrices[area] = {
            'confusion_matrix': svm_result['confusion_matrix'],
            'labels': svm_result['labels']
        }
        
        print(f"      Accuracy: {mean_accuracy:.3f}, p-value: {p_value:.4f}")
    
    # Store fish results
    fish_key = f"{fb_id}_{hb_id}"
    all_svm_results[fish_key] = fish_results
    all_svm_confusion_matrices[fish_key] = fish_conf_matrices
    
    print(f"  Completed {len(fish_results)} areas")

print(f"\n{'='*80}")
print(f"✓ SVM analysis complete")
print(f"{'='*80}")


### Statistical Combination


In [ ]:
# Reorganize by area
all_areas_svm = {}

for fish_id, fish_results in all_svm_results.items():
    for area_name, area_results in fish_results.items():
        mean_score = area_results['mean_score']
        p_value = area_results['p_value']
        
        if area_name not in all_areas_svm:
            all_areas_svm[area_name] = []
        
        all_areas_svm[area_name].append((fish_id, mean_score, p_value))

print(f"\nCombining SVM statistics...")
print(f"  Total areas: {len(all_areas_svm)}")

# Fisher's + FDR
df_svm_combined = combine_fish_statistics(all_areas_svm, fdr_alpha=FDR_ALPHA)

# Display
print(f"\n{'='*80}")
print("SVM CLASSIFICATION RESULTS")
print(f"{'='*80}")
sig_areas_svm = df_svm_combined[df_svm_combined['is_significant']]
print(f"\nSignificant areas (FDR < {FDR_ALPHA}):")
print(f"  {len(sig_areas_svm)}/{len(df_svm_combined)} areas")

if len(sig_areas_svm) > 0:
    print(f"\nTop 10:")
    print(sig_areas_svm[['area', 'mean_accuracy', 'fdr_corrected']].head(10).to_string(index=False))

print(f"\nAccuracy statistics:")
print(f"  Mean: {df_svm_combined['mean_accuracy'].mean():.3f}")
print(f"  Range: {df_svm_combined['mean_accuracy'].min():.3f} - {df_svm_combined['mean_accuracy'].max():.3f}")
print(f"  Chance: {1/len(KEY_STIMULI):.3f}")


### Save SVM Results


In [ ]:
# Save
output_file_svm = f"{OUTPUT_DIR}/svm_classification_7odors_refactored.csv"
df_svm_combined.to_csv(output_file_svm, index=False)

print(f"\n✓ SVM results saved to:")
print(f"  {output_file_svm}")
print(f"  Rows: {len(df_svm_combined)}")
print(f"  Significant: {df_svm_combined['is_significant'].sum()}")


## 6. SVM Confusion Matrix Analysis

**Same template matching style as LDA analysis**


### Aggregate SVM Confusion Matrix


In [ ]:
# Aggregate
aggregate_svm_cm = np.zeros((len(KEY_STIMULI), len(KEY_STIMULI)))
n_svm_matrices = 0

for fish_id, fish_conf_dict in all_svm_confusion_matrices.items():
    for area, conf_data in fish_conf_dict.items():
        cm = conf_data['confusion_matrix']
        aggregate_svm_cm += cm
        n_svm_matrices += 1

aggregate_svm_cm_avg = aggregate_svm_cm / n_svm_matrices

print(f"Aggregated {n_svm_matrices} SVM confusion matrices")


### Visualize Aggregate SVM Confusion Matrix


In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(10, 8))

im = plot_confusion_matrix_styled(
    aggregate_svm_cm_avg,
    list(KEY_STIMULI),
    title="SVM: Aggregate Confusion Matrix\n(All Fish & Areas)",
    ax=ax
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Prediction Probability', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Aggregate SVM confusion matrix plotted")


### Per-Fish SVM Confusion Matrices


In [ ]:
# Per-fish
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for fish_idx, (fish_key, fish_conf_dict) in enumerate(all_svm_confusion_matrices.items()):
    ax = axes[fish_idx]
    
    fish_cm = np.zeros((len(KEY_STIMULI), len(KEY_STIMULI)))
    n_areas = 0
    
    for area, conf_data in fish_conf_dict.items():
        fish_cm += conf_data['confusion_matrix']
        n_areas += 1
    
    fish_cm_avg = fish_cm / n_areas
    
    plot_confusion_matrix_styled(
        fish_cm_avg,
        list(KEY_STIMULI),
        title=f"{simplified_fish_ids[fish_key]} (n={n_areas})",
        ax=ax
    )

fig.suptitle('SVM: Per-Fish Confusion Matrices', 
            fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()

print(f"\n✓ Per-fish SVM confusion matrices plotted")


### SVM Confusion Analysis


In [ ]:
# Analyze
print("="*80)
print("SVM CONFUSION ANALYSIS")
print("="*80)

svm_analysis = analyze_confusion_pairs(aggregate_svm_cm_avg, list(KEY_STIMULI))

print("\nTop 10 Confusion Pairs:")
print("-"*80)
for i, (true_stim, pred_stim, prob) in enumerate(svm_analysis['top_confusions'], 1):
    print(f"{i:2d}. {true_stim:12s} → {pred_stim:12s}  (p={prob:.3f})")

print("\n" + "="*80)
print("VALENCE-BASED ANALYSIS")
print("="*80)
print(f"\nWithin-attractive: {svm_analysis['within_attr_avg']:.3f}")
print(f"Within-aversive:   {svm_analysis['within_aver_avg']:.3f}")
print(f"Cross-valence:     {svm_analysis['cross_avg']:.3f}")

if svm_analysis['within_attr_avg'] > svm_analysis['cross_avg'] or \
   svm_analysis['within_aver_avg'] > svm_analysis['cross_avg']:
    print("\n✓ Confusions MORE common within same valence")
    print("  → SVM encodes valence structure")
else:
    print("\n✓ Confusions NOT structured by valence")
    print("  → SVM encodes odor identity independently")


### Save SVM Confusion Matrix


In [ ]:
# Save
cm_svm_file = f"{OUTPUT_DIR}/svm_aggregate_confusion_matrix.csv"
cm_svm_df = pd.DataFrame(
    aggregate_svm_cm_avg,
    index=KEY_STIMULI,
    columns=KEY_STIMULI
)
cm_svm_df.to_csv(cm_svm_file)

print(f"\n✓ SVM confusion matrix saved:")
print(f"  {cm_svm_file}")
print("\n" + "="*80)
print("SVM ANALYSIS COMPLETE")
print("="*80)


## 7. Method Comparison

Compare LDA vs SVM classification performance


In [ ]:
# Compare methods
print("="*80)
print("METHOD COMPARISON: LDA vs SVM")
print("="*80)

print(f"\nLDA:")
print(f"  Mean accuracy: {df_lda_combined['mean_accuracy'].mean():.3f}")
print(f"  Significant areas: {df_lda_combined['is_significant'].sum()}/{len(df_lda_combined)}")
print(f"  Best area: {df_lda_combined.iloc[0]['area']} ({df_lda_combined.iloc[0]['mean_accuracy']:.3f})")

print(f"\nSVM:")
print(f"  Mean accuracy: {df_svm_combined['mean_accuracy'].mean():.3f}")
print(f"  Significant areas: {df_svm_combined['is_significant'].sum()}/{len(df_svm_combined)}")
print(f"  Best area: {df_svm_combined.iloc[0]['area']} ({df_svm_combined.iloc[0]['mean_accuracy']:.3f})")

print(f"\nChance: {1/len(KEY_STIMULI):.3f}")

# Determine winner
lda_mean = df_lda_combined['mean_accuracy'].mean()
svm_mean = df_svm_combined['mean_accuracy'].mean()

print(f"\n{'='*80}")
if lda_mean > svm_mean:
    print(f"✓ LDA performs better on average (+{lda_mean-svm_mean:.3f})")
elif svm_mean > lda_mean:
    print(f"✓ SVM performs better on average (+{svm_mean-lda_mean:.3f})")
else:
    print("✓ LDA and SVM perform equally")
print("="*80)


## 8. Summary

### Analysis Complete!

**LDA Classification:**
- PCA + LDA pipeline
- Confusion matrices in template matching style
- Statistical significance tested

**SVM Classification:**
- LinearSVC classifier
- Same confusion matrix analysis
- Comparable performance

### Output Files:
1. LDA results CSV
2. LDA confusion matrix CSV
3. SVM results CSV
4. SVM confusion matrix CSV

### Key Findings:
- Compare which method performs better
- Examine confusion patterns
- Identify best brain areas for classification


In [ ]:
# Final summary
print("="*80)
print("LDA/SVM CLASSIFICATION ANALYSIS COMPLETE")
print("="*80)
print(f"\n✓ LDA Analysis:")
print(f"    Areas: {len(df_lda_combined)}")
print(f"    Significant: {df_lda_combined['is_significant'].sum()}")
print(f"    Mean accuracy: {df_lda_combined['mean_accuracy'].mean():.3f}")
print(f"\n✓ SVM Analysis:")
print(f"    Areas: {len(df_svm_combined)}")
print(f"    Significant: {df_svm_combined['is_significant'].sum()}")
print(f"    Mean accuracy: {df_svm_combined['mean_accuracy'].mean():.3f}")
print(f"\n✓ All confusion matrices styled consistently")
print(f"✓ Output files saved to {OUTPUT_DIR}")
print("="*80)
